In [ ]:
!pip install --upgrade transformers huggingface_hub datasets -q

In [ ]:
import time
import json
import re
from copy import deepcopy
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1) LLM Distillation

**The problem**
Large reasoning models (DeepSeek-R1 at 671B, GPT-5 at unknown size) are expensive to serve. A single inference can cost 10–100x more than a small model. But small models trained from scratch with GRPO plateau at much lower reasoning ability than large ones — they simply don't have enough capacity to discover sophisticated reasoning patterns on their own.

**The solution: distillation**
Instead of asking the small model to discover reasoning from scratch, we show it how the large model reasons. The small model learns to imitate the large model's reasoning traces — its chain-of-thought, self-verification, and problem-solving strategies.

![Knowledge Distillation](https://upload.wikimedia.org/wikipedia/commons/b/bd/Knowledge-distillation-cv.png)

## Three Distillation Methods

| Method | What the student learns from | Requires teacher logits? | Quality |
|--------|----------------------------|-------------------------|---------|
| Response-level SFT | Teacher's text output (reasoning traces) | No — only text | Good |
| Logit-level KD | Teacher's probability distribution (soft targets) | Yes — need forward pass | Better |
| Rejection sampling + SFT | Filtered teacher outputs (only correct ones) | No — only text | Best |

## 1.1 Setup

In [ ]:
# We use small, modern instruction-tuned models to simulate teacher-student distillation.
# The teacher needs to be capable of basic reasoning to generate correct traces.
TEACHER_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
STUDENT_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load teacher model in evaluation mode (we only do forward passes to get traces/logits)
teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_NAME)
teacher_model = AutoModelForCausalLM.from_pretrained(TEACHER_NAME, torch_dtype=torch.bfloat16).to(device)
teacher_model.eval()
for p in teacher_model.parameters():
    p.requires_grad = False

# Load student model
student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_NAME)
student_model = AutoModelForCausalLM.from_pretrained(STUDENT_NAME, torch_dtype=torch.bfloat16).to(device)

# Ensure pad tokens are set for batching
for tok in [teacher_tokenizer, student_tokenizer]:
    if tok.pad_token_id is None:
        tok.pad_token_id = tok.eos_token_id

print(f"Teacher: {TEACHER_NAME} ({sum(p.numel() for p in teacher_model.parameters()):,} params)")
print(f"Student: {STUDENT_NAME} ({sum(p.numel() for p in student_model.parameters()):,} params)")
print(f"Compression ratio: {sum(p.numel() for p in teacher_model.parameters()) / sum(p.numel() for p in student_model.parameters()):.1f}x")
print(f"Device: {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Teacher: Qwen/Qwen2.5-1.5B-Instruct (1,543,714,304 params)
Student: Qwen/Qwen2.5-0.5B-Instruct (494,032,768 params)
Compression ratio: 3.1x
Device: cuda


The first step in distillation is generating training data from the teacher. We give the teacher a set of problems, let it produce chain-of-thought reasoning, and collect the full output as training data for the student.

This is exactly what DeepSeek did: they used DeepSeek-R1 to generate ~800K reasoning traces, then fine-tuned smaller Qwen and Llama models on these traces.

In [ ]:
PROBLEMS = [
    {"prompt": "What is 15 + 27? Think step by step, then give your answer.", "answer": "42"},
    {"prompt": "What is 8 * 7? Think step by step, then give your answer.", "answer": "56"},
    {"prompt": "What is 100 - 37? Think step by step, then give your answer.", "answer": "63"},
    {"prompt": "What is 144 / 12? Think step by step, then give your answer.", "answer": "12"},
    {"prompt": "What is 23 + 19? Think step by step, then give your answer.", "answer": "42"},
    {"prompt": "What is 9 * 9? Think step by step, then give your answer.", "answer": "81"},
    {"prompt": "What is 256 - 128? Think step by step, then give your answer.", "answer": "128"},
    {"prompt": "What is 72 / 8? Think step by step, then give your answer.", "answer": "9"},
]

@torch.no_grad()
def generate_teacher_traces(
    model,
    tokenizer,
    problems: List[dict],
    num_traces_per_problem: int = 4,
    max_new_tokens: int = 128,
    temperature: float = 0.7,
) -> List[dict]:
    traces = []
    model.eval()

    for problem in problems:
        # Tokenize prompt
        # (1, seq_len)
        inputs = tokenizer(problem["prompt"], return_tensors="pt").to(device)

        for _ in range(num_traces_per_problem):
            # Generate reasoning trace
            # (1, seq_len) -> (1, seq_len + max_new_tokens)
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=True,
                top_p=0.95,
                pad_token_id=tokenizer.pad_token_id,
            )

            # Extract generated portion
            # (seq_len + max_new_tokens) -> (max_new_tokens)
            generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
            trace_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

            ground_truth = problem["answer"]
            is_correct = ground_truth in trace_text

            traces.append({
                "prompt": problem["prompt"],
                "trace": trace_text,
                "answer": ground_truth,
                "is_correct": is_correct,
            })

    return traces

print("Generating teacher traces...")
all_traces = generate_teacher_traces(
    teacher_model,
    teacher_tokenizer,
    PROBLEMS,
    num_traces_per_problem=4,
    max_new_tokens=128,
    temperature=0.7,
)

correct = sum(1 for t in all_traces if t["is_correct"])
print(f"Generated {len(all_traces)} traces, {correct} correct ({correct/len(all_traces):.0%})")
print(f"\nSample Trace: {all_traces[0]}")

Generating teacher traces...
Generated 32 traces, 17 correct (53%)

Sample Trace: {'prompt': 'What is 15 + 27? Think step by step, then give your answer.', 'trace': ' Step 1: Identify the numbers being added.\nIn this case, we have two whole numbers:\n- 15\n- 27\n\nStep 2: Determine whether to add or subtract.\nSince both numbers are positive (not negative), we can simply add them together.\n\nStep 3: Perform the addition.\nTo add 15 and 27, start with the ones place:\n15 + 0 = 15\nNow move on to the tens place:\n1 + 2 = 3\nSo when you combine these values, you get:\n15 + 27 = 42\n\n', 'answer': '42', 'is_correct': True}


## 1.2 Response-Level Distillation (SFT on Teacher Traces)

The simplest form of distillation: treat the teacher's reasoning traces as supervised training data and fine-tune the student with standard cross-entropy loss.

In [ ]:
class DistillationDataset(Dataset):
    def __init__(self, traces: List[dict], tokenizer, max_length: int = 512):
        self.samples = []
        for trace in traces:
            # Tokenize just the prompt to know its length
            prompt_ids = tokenizer(trace["prompt"], add_special_tokens=True)["input_ids"]

            # Tokenize the full sequence (prompt + teacher's reasoning trace)
            full_text = trace["prompt"] + trace["trace"]
            full = tokenizer(
                full_text, add_special_tokens=True, truncation=True, max_length=max_length
            )

            # Create labels for causal language modeling.
            # We want the student to learn to predict the trace, NOT the prompt.
            # So we mask out the prompt tokens using -100 (PyTorch's default ignore_index for CrossEntropyLoss).
            prompt_len = len(prompt_ids)
            labels = [-100] * prompt_len + full["input_ids"][prompt_len:]
            labels = labels[:len(full["input_ids"])]

            self.samples.append({
                "input_ids": torch.tensor(full["input_ids"]),
                "attention_mask": torch.tensor(full["attention_mask"]),
                "labels": torch.tensor(labels),
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def collate_fn(batch, pad_id=0):
    """Pads a batch of varying length sequences to the maximum length in the batch."""
    max_len = max(b["input_ids"].size(0) for b in batch)
    padded = {key: [] for key in batch[0]}

    for b in batch:
        pad_size = max_len - b["input_ids"].size(0)
        for key in b:
            # Pad input_ids with pad_token_id, labels with -100, and attention_mask with 0
            pad_val = pad_id if key == "input_ids" else (-100 if key == "labels" else 0)
            # F.pad pads from the end: (left_pad, right_pad)
            padded[key].append(F.pad(b[key], (0, pad_size), value=pad_val))

    return {k: torch.stack(v) for k, v in padded.items()}

In [ ]:
def train_sft_distillation(
    student_model,
    tokenizer,
    traces: List[dict],
    num_epochs: int = 1,
    batch_size: int = 2,
    lr: float = 5e-5,
    max_length: int = 256,
) -> List[dict]:
    """
    Fine-tunes the student model using Supervised Fine-Tuning (SFT) on the teacher's reasoning traces.
    """
    dataset = DistillationDataset(traces, tokenizer, max_length)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=lambda b: collate_fn(b, pad_id=tokenizer.pad_token_id),
    )

    optimizer = torch.optim.AdamW(student_model.parameters(), lr=lr)
    # CrossEntropyLoss automatically ignores tokens with label = -100
    loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
    all_metrics = []
    student_model.train()

    for epoch in range(num_epochs):
        epoch_loss = 0.0
        num_batches = 0

        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            # Forward pass: get student predictions
            outputs = student_model(
                input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]
            )

            # Shift logits and labels by 1.
            # At position t, the model predicts the token at t+1.
            # Therefore, we align logits[:-1] with labels[1:]
            shift_logits = outputs.logits[:, :-1, :].contiguous()
            shift_labels = batch["labels"][:, 1:].contiguous()

            # Flatten the batch and sequence dimensions to compute loss
            # shift_logits: (batch * seq_len, vocab_size)
            # shift_labels: (batch * seq_len)
            loss = loss_fn(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            )

            # Standard optimization step
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student_model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_loss += loss.item()
            num_batches += 1

        avg_loss = epoch_loss / max(1, num_batches)
        all_metrics.append({"epoch": epoch, "loss": avg_loss})
        print(f"  Epoch {epoch}: loss={avg_loss:.4f}")

    return all_metrics

student_model = AutoModelForCausalLM.from_pretrained(STUDENT_NAME).to(device)
sft_metrics = train_sft_distillation(student_model, student_tokenizer, all_traces, num_epochs=1)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  Epoch 0: loss=1.0298


## 1.3 Logit-Level Distillation (KL Divergence)

- Response-level distillation only uses the teacher's **text output**.
- But the teacher knows more than just the text — at every token position, it has a full probability distribution over the entire vocabulary.

In [ ]:
def distillation_loss(student_logits, teacher_logits, labels, temperature=2.0, alpha=0.5):

    """Compute combined distillation loss: KL divergence + cross-entropy.

    The KL term teaches the student to match the teacher's full distribution.
    The CE term teaches the student to get the right answer.
    Alpha balances between the two.

    Args:
        student_logits: (batch_num, seq_len, vocab_size) — student's raw predictions
        teacher_logits: (batch_num, seq_len, vocab_size) — teacher's raw predictions
        labels:         (batch_num, seq_len) — ground truth token IDs (-100 for prompt)
        temperature:    Softening temperature (higher = softer distributions)
        alpha:          Balance between KL (alpha) and CE (1 - alpha)

    Returns:
        loss: scalar, metrics: dict
    """

    # Compute soft probability distributions at elevated temperature
    # Temperature > 1 flattens the distributions, revealing "dark knowledge" (secondary probabilities)
    # (batch_num, seq_len, vocab_size) -> (batch_num, seq_len, vocab_size)
    teacher_soft = F.log_softmax(teacher_logits / temperature, dim=-1)
    student_soft = F.log_softmax(student_logits / temperature, dim=-1)

    # KL Divergence loss calculates how much the student's probability distribution diverges from the teacher's
    # We multiply by temperature^2 to scale the gradients back up since temperature scaled down the logits
    kl_loss = (
        F.kl_div(student_soft, teacher_soft, log_target=True, reduction="batchmean") *
         (temperature ** 2)
    )

    # Standard Cross-Entropy loss against the actual token labels (ignoring prompt tokens via -100)
    ce_loss = F.cross_entropy(
        student_logits.reshape(-1, student_logits.size(-1)), labels.reshape(-1), ignore_index=-100
    )

    # Combine the two losses using alpha
    loss = alpha * kl_loss + (1 - alpha) * ce_loss
    return loss, {"loss": loss.item(), "kl_loss": kl_loss.item(), "ce_loss": ce_loss.item()}

def train_logit_distillation(teacher_model, student_model, tokenizer, traces, num_epochs=1):
    dataset = DistillationDataset(traces, tokenizer)
    loader = DataLoader(
        dataset,
        batch_size=4,
        shuffle=True,
        collate_fn=lambda b: collate_fn(b, pad_id=tokenizer.pad_token_id)
    )
    optimizer = torch.optim.AdamW(student_model.parameters(), lr=5e-5)

    for epoch in range(num_epochs):
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            with torch.no_grad():
                # Teacher forward pass to get target logits.
                # (batch_num, seq_len) -> (batch_num, seq_len, vocab_size)
                teacher_outputs = teacher_model(
                    input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]
                )
                # Shift teacher logits by 1 to align with next-token prediction
                teacher_logits = teacher_outputs.logits[:, :-1, :].contiguous()

            # Student forward pass to get predicted logits
            # (batch_num, seq_len) -> (batch_num, seq_len, vocab_size)
            student_outputs = student_model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            student_logits = student_outputs.logits[:, :-1, :].contiguous()

            # Shift labels by 1
            shift_labels = batch["labels"][:, 1:].contiguous()

            # Compute the combined loss
            loss, metrics = distillation_loss(student_logits, teacher_logits, shift_labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f"  Epoch {epoch}: loss={metrics['loss']:.4f}")

# === TEST RUN ===
student_model = AutoModelForCausalLM.from_pretrained(STUDENT_NAME).to(device)
kd_metrics = train_logit_distillation(teacher_model, student_model, student_tokenizer, all_traces, num_epochs=1)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  Epoch 0: loss=106.5000


## 1.4 Rejection Sampling
Not all teacher traces are equally good. Some contain wrong answers, sloppy reasoning, or language mixing. Training the student on bad traces teaches it bad habits.

**Rejection sampling** addresses this: generate many traces, score them, and only keep the best ones. This is exactly what DeepSeek did:

1. Generate many traces from the R1 checkpoint
2. Score each trace for correctness (does it get the right answer?)
3. Optionally score for quality (using DeepSeek-V3 as a judge)
4. Keep only the traces that pass both filters
5. SFT the student on the filtered, high-quality traces

This is simple but surprisingly effective — the quality of the training data matters more than the training method.

In [ ]:
def rejection_sample(
    traces: List[dict],
    require_correct: bool = True,
    max_length: int = None,
    min_length: int = None,
) -> List[dict]:

    """Filter teacher traces by quality criteria."""

    filtered = []
    for trace in traces:
        # Filter by correctness
        if require_correct and not trace["is_correct"]:
            continue

        # Filter by length
        trace_len = len(trace["trace"])
        if max_length and trace_len > max_length:
            continue
        if min_length and trace_len < min_length:
            continue

        filtered.append(trace)

    return filtered

# Apply rejection sampling
filtered_traces = rejection_sample(
    all_traces,
    require_correct=True,
    min_length=10,
)

print(f"Before filtering: {len(all_traces)} traces")
print(f"After filtering: {len(filtered_traces)} traces")
print(f"Kept: {len(filtered_traces)/max(1, len(all_traces)):.0%}")

if len(filtered_traces) == 0:
    print("\nNo correct traces found — using all traces for demo purposes")
    filtered_traces = all_traces

# Train on filtered traces
student_model = AutoModelForCausalLM.from_pretrained(STUDENT_NAME).to(device)
rs_metrics = train_sft_distillation(
    student_model,
    student_tokenizer,
    filtered_traces,
    num_epochs=1,
    batch_size=4,
    lr=5e-5,
)

Before filtering: 32 traces
After filtering: 17 traces
Kept: 53%


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  Epoch 0: loss=0.9234


## 1.5 Evaluation

After distillation, we test whether the student learned to produce reasoning traces similar to the teacher. We check:

1. **Perplexity on held-out teacher traces** — does the student assign high probability to the teacher's reasoning?
2. **Generation quality** — can the student produce coherent chain-of-thought?
3. **Answer accuracy** — does the student get the right answer?

In [ ]:
@torch.no_grad()
def evaluate_student(
    model,
    tokenizer,
    problems: List[dict],
    max_new_tokens: int = 128,
    temperature: float = 0.7,
) -> dict:
    """Evaluate the distilled student model on reasoning problems."""
    model.eval()
    correct = 0
    total = 0
    sample_outputs = []

    for problem in problems:
        # Tokenize prompt
        # (1, seq_len)
        inputs = tokenizer(problem["prompt"], return_tensors="pt").to(device)

        # Generate student response
        # (1, seq_len) -> (1, seq_len + max_new_tokens)
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
        )

        # Decode generated response
        generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
        response = tokenizer.decode(generated_ids, skip_special_tokens=True)

        # Check correctness
        is_correct = problem["answer"] in response
        if is_correct:
            correct += 1
        total += 1

        sample_outputs.append({
            "prompt": problem["prompt"],
            "response": response[:200],
            "correct": is_correct,
        })

    accuracy = correct / max(1, total)
    avg_length = sum(len(s["response"]) for s in sample_outputs) / max(1, len(sample_outputs))

    return {
        "accuracy": accuracy,
        "avg_response_length": avg_length,
        "samples": sample_outputs,
    }

# Evaluate the last trained student
eval_results = evaluate_student(student_model, student_tokenizer, PROBLEMS[:4])
print(f"Student accuracy: {eval_results['accuracy']:.0%}")
print(f"Avg response length: {eval_results['avg_response_length']:.0f} chars")
print(f"\nSample outputs:")
for s in eval_results["samples"]:
    status = "✓" if s["correct"] else "✗"
    print(f"  {status} {s['prompt'][:40]}...")
    print(f"    → {s['response'][:100]}...")

Student accuracy: 100%
Avg response length: 200 chars

Sample outputs:
  ✓ What is 15 + 27? Think step by step, the...
    →  To find the sum of 15 and 27, you can follow these steps:

1. **Understand Addition**:
   Addition ...
  ✓ What is 8 * 7? Think step by step, then ...
    →  To find the product of 8 and 7, follow these steps:

1. **Understand multiplication**: Multiplicati...
  ✓ What is 100 - 37? Think step by step, th...
    →  To find the difference between 100 and 37, we can break it down into simple steps:

1. **Understand...
  ✓ What is 144 / 12? Think step by step, th...
    →  To find the value of \( \frac{144}{12} \), you can break it down into steps:

1. **Understand what ...


# 2) On-Policy Distillation (OPD)

![](https://raw.githubusercontent.com/NVIDIA-NeMo/.github/discussions/discussions/posts/on-policy-distillation/assets/comparison.png)

The three methods above are all **off-policy** — the training data comes from the teacher's distribution, not the student's. The student only sees the teacher's reasoning traces, never its own mistakes.

This causes **exposure bias**: during training, the student always sees perfect teacher tokens as context. But at inference, it sees its own (imperfect) tokens. If the student makes a small error early on, it enters a state it never encountered during training, and errors compound.

**On-policy distillation fixes this.** Instead of training on teacher-generated text, the student generates its own responses, and the teacher provides token-level supervision on those student-generated tokens.

This is the approach Qwen3 used, and it achieves comparable performance to GRPO at 1/10th the compute cost.


In [ ]:
def train_on_policy_distillation(
    teacher_model,
    student_model,
    tokenizer,
    prompts: List[str],
    num_epochs: int = 1,
    lr: float = 5e-5,
    max_new_tokens: int = 64,
    temperature: float = 1.0,
    kl_type: str = "reverse",
) -> List[dict]:

    optimizer = torch.optim.AdamW(student_model.parameters(), lr=lr)
    all_metrics = []
    teacher_model.eval()

    for epoch in range(num_epochs):
        epoch_loss = 0.0
        num_steps = 0

        for prompt in prompts:
            student_model.eval()
            # Tokenize prompt to get its exact sequence length
            # (1, seq_len)
            prompt_ids = tokenizer(prompt, return_tensors="pt").to(device)
            prompt_len = prompt_ids["input_ids"].shape[1]

            with torch.no_grad():
                # Step 1: Student generates autoregressively (This is "on-policy")
                # The student creates its own reasoning path, which may contain mistakes.
                # (1, seq_len) -> (1, seq_len + max_new_tokens)
                student_output_ids = student_model.generate(
                    **prompt_ids,
                    max_new_tokens=max_new_tokens,
                    temperature=temperature,
                    do_sample=True,
                    top_p=0.95,
                    pad_token_id=tokenizer.pad_token_id,
                )

            full_ids = student_output_ids
            total_len = full_ids.shape[1]

            # Build response mask: 1 for student-generated tokens, 0 for prompt
            # We only want to compute loss on the generated response, not the prompt.
            # (1, total_len)
            response_mask = torch.zeros(1, total_len, device=device)
            response_mask[:, prompt_len:] = 1.0

            # Step 2: Teacher evaluates the student's generation.
            # The teacher does a single forward pass over the student's entire output to score it.
            # (1, total_len) -> (1, total_len, vocab_size)
            with torch.no_grad():
                teacher_logits = teacher_model(full_ids).logits
            teacher_logits_shifted = teacher_logits[:, :-1, :].contiguous()

            # Step 3: Student evaluates its own generation (requires gradients this time)
            student_model.train()
            # (1, total_len) -> (1, total_len, vocab_size)
            student_logits = student_model(full_ids).logits
            student_logits_shifted = student_logits[:, :-1, :].contiguous()
            mask_shifted = response_mask[:, 1:].contiguous()

            # Convert logits to probabilities and log-probabilities
            teacher_probs = F.softmax(teacher_logits_shifted / temperature, dim=-1)
            student_log_probs = F.log_softmax(student_logits_shifted / temperature, dim=-1)

            # Compute Token-Level KL Divergence
            if kl_type == "reverse":
                # Reverse KL: penalizes student for giving probability to tokens teacher thinks are bad (mode-seeking)
                student_probs = F.softmax(student_logits_shifted / temperature, dim=-1)
                kl_per_token = (student_probs * (student_probs.log() - teacher_probs.log())).sum(dim=-1)
            else:
                # Forward KL: penalizes student for missing tokens the teacher thinks are good (mean-seeking)
                kl_per_token = (teacher_probs * (teacher_probs.log() - student_log_probs)).sum(dim=-1)

            # Apply mask to zero out KL penalty on prompt tokens
            masked_kl = kl_per_token * mask_shifted
            # Average the loss only over the generated tokens (where mask == 1)
            loss = (temperature ** 2) * masked_kl.sum() / mask_shifted.sum().clamp_min(1)

            # Backprop
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student_model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_loss += loss.item()
            num_steps += 1

        avg_loss = epoch_loss / max(1, num_steps)
        all_metrics.append({"epoch": epoch, "loss": avg_loss})
        print(f"  Epoch {epoch}: loss={avg_loss:.4f}")

    return all_metrics

student_model = AutoModelForCausalLM.from_pretrained(STUDENT_NAME).to(device)
opd_prompts = [p["prompt"] for p in PROBLEMS]
opd_metrics = train_on_policy_distillation(
    teacher_model, student_model, student_tokenizer, opd_prompts, num_epochs=1
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  Epoch 0: loss=0.7380


# 3) Reinforcement Distillation

Standard rejection sampling keeps only the correct traces and throws away the incorrect ones. But the incorrect traces contain useful information too — they show the model what NOT to do.

**REDI (Reinforcement Distillation)** uses both:
- Correct traces → positive examples (increase probability)
- Incorrect traces → negative examples (decrease probability)

This is essentially **DPO applied to distillation data**:


In [ ]:
def build_redi_pairs(traces: List[dict]) -> List[dict]:
    """Build DPO-style preference pairs from teacher traces."""
    by_prompt = {}
    # Group traces by prompt and separate them into correct (chosen) and incorrect (rejected)
    for t in traces:
        key = t["prompt"]
        if key not in by_prompt:
            by_prompt[key] = {"correct": [], "incorrect": []}
        if t["is_correct"]:
            by_prompt[key]["correct"].append(t["trace"])
        else:
            by_prompt[key]["incorrect"].append(t["trace"])

    pairs = []
    # Create pairs of (chosen, rejected) responses for the same prompt
    for prompt, groups in by_prompt.items():
        if not groups["correct"] or not groups["incorrect"]:
            continue
        for chosen in groups["correct"]:
            for rejected in groups["incorrect"]:
                pairs.append({
                    "prompt": prompt,
                    "chosen": chosen,
                    "rejected": rejected,
                })
                # Taking only one pair per prompt for simplicity
                break
            break
    return pairs

def get_sequence_logps(model, tokenizer, prompt: str, response: str, max_length: int = 512) -> torch.Tensor:
    """Compute the sum of log-probabilities for a response given a prompt.
    This represents how likely the model thinks this specific response is.
    """
    prompt_ids = tokenizer(prompt, add_special_tokens=True)["input_ids"]
    full = tokenizer(
        prompt + response,
        add_special_tokens=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    ).to(device)
    prompt_len = len(prompt_ids)

    # Forward pass (with or without gradients depending on model mode)
    with torch.no_grad() if not model.training else torch.enable_grad():
        logits = model(**full).logits

    # Shift logits to align with next-token prediction targets
    shifted_logits = logits[:, :-1, :]
    shifted_targets = full["input_ids"][:, 1:]

    # Get log-softmax probabilities for all tokens in vocabulary
    log_probs = F.log_softmax(shifted_logits, dim=-1)
    # Gather only the probabilities of the actual tokens in the sequence
    per_token_logps = torch.gather(log_probs, dim=2, index=shifted_targets.unsqueeze(2)).squeeze(2)

    # We only care about the likelihood of the *response*, so we mask out the prompt
    mask = torch.zeros_like(per_token_logps)
    mask[:, prompt_len - 1:] = 1.0

    # Return the sum of log probabilities for the generated response
    return (per_token_logps * mask).sum()

def train_redi(student_model, tokenizer, pairs: List[dict], num_epochs: int = 1, lr: float = 5e-5, beta: float = 0.1) -> List[dict]:
    """Train using Direct Preference Optimization (DPO) on the distillation pairs."""
    optimizer = torch.optim.AdamW(student_model.parameters(), lr=lr)
    all_metrics = []
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        epoch_acc = 0.0
        num_steps = 0
        for pair in pairs:
            student_model.train()

            # Get likelihood of the correct trace
            chosen_logps = get_sequence_logps(student_model, tokenizer, pair["prompt"], pair["chosen"])
            # Get likelihood of the incorrect trace
            rejected_logps = get_sequence_logps(student_model, tokenizer, pair["prompt"], pair["rejected"])

            # DPO Loss: We want to maximize the difference between chosen and rejected log-probs.
            # Beta controls how much we penalize deviating from the reference model (omitted here for simplicity)
            logits = beta * (chosen_logps - rejected_logps)
            # Negative log-sigmoid creates a loss that pushes `chosen_logps > rejected_logps`
            loss = -F.logsigmoid(logits)

            with torch.no_grad():
                # Calculate accuracy: Did the model assign higher probability to the correct trace?
                acc = (chosen_logps > rejected_logps).float().item()

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student_model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_loss += loss.item()
            epoch_acc += acc
            num_steps += 1

        avg_loss = epoch_loss / max(1, num_steps)
        avg_acc = epoch_acc / max(1, num_steps)
        all_metrics.append({"epoch": epoch, "loss": avg_loss, "preference_acc": avg_acc})
        print(f"  Epoch {epoch}: loss={avg_loss:.4f} preference_acc={avg_acc:.0%}")
    return all_metrics

redi_pairs = build_redi_pairs(all_traces)
print(f"\nBuilt {len(redi_pairs)} REDI preference pairs")

if len(redi_pairs) > 0:
    student_model = AutoModelForCausalLM.from_pretrained(STUDENT_NAME).to(device)
    redi_metrics = train_redi(student_model, student_tokenizer, redi_pairs, num_epochs=1, lr=5e-5, beta=0.1)
else:
    print("No pairs with both correct and incorrect traces — skipping REDI demo")


Built 5 REDI preference pairs


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  Epoch 0: loss=8.1567 preference_acc=20%


The most effective approach is to combine distillation with RL:

```
Step 1: Distill — SFT on teacher traces (gives strong baseline quickly)
Step 2: GRPO — RL on the distilled student (pushes beyond teacher's traces)
```

- **Distillation alone** is bounded by the teacher's traces.
- **GRPO alone** on a small model plateaus at low performance because it can't discover complex reasoning.
- **Distillation then GRPO** gives the student a strong starting point, then RL pushes it further.

Comparison and Practical Guidelines

| Method | When to use | Cost | Quality |
|--------|------------|------|---------|
| Response-level SFT | Default choice. No teacher logit access. Fast. | Low | Good |
| Logit-level KD | You have full access to teacher weights. | Medium | Better |
| Rejection sampling + SFT | You can generate many traces cheaply. | Low | Best |

**DeepSeek's actual recipe:** Rejection sampling + SFT. They used DeepSeek-V3 as a judge for quality scoring.